### `Generate Key`

In [ ]:
import random
import string


def generate_key() -> str:
    length = random.randint(4, 8)
    return "".join(random.choices(string.ascii_uppercase + string.digits, k=length))


key = generate_key()
print(f"Generated key: {key}")

### `Encryption`

In [ ]:
import os
import string

CHARS = string.ascii_uppercase + string.digits + "@#%&*()_+-={}[]:;\"'<>,.?! \n\t"


def generate_key_data(key: str) -> tuple[list[list[str]], dict[str, tuple[int, int]]]:
    key = key.upper()
    seen_chars = []
    seen_set = set()
    for ch in key + CHARS:
        if ch in CHARS and ch not in seen_set:
            seen_chars.append(ch)
            seen_set.add(ch)
    table = [seen_chars[i * 8 : (i + 1) * 8] for i in range(8)]
    lookup = {ch: (r, c) for r, row in enumerate(table) for c, ch in enumerate(row)}
    return table, lookup


def prepare_text(text: str) -> list[str]:
    filtered = [ch for ch in text.upper() if ch in CHARS]
    digrams = []
    i = 0
    while i < len(filtered):
        a = filtered[i]
        if i + 1 < len(filtered):
            b = filtered[i + 1]
            if a == b:
                filler = "X" if a != "X" else "Y"
                digrams.extend([a, filler])
                i += 1
            else:
                digrams.extend([a, b])
                i += 2
        else:
            filler = "X" if a != "X" else "Y"
            digrams.extend([a, filler])
            i += 1
    return digrams


def encrypt_digrams(
    table: list[list[str]], lookup: dict[str, tuple[int, int]], digrams: list[str]
) -> str:
    result = []
    for i in range(0, len(digrams), 2):
        a, b = digrams[i], digrams[i + 1]
        ra, ca = lookup[a]
        rb, cb = lookup[b]
        if ra == rb:
            result.append(table[ra][(ca + 1) % 8])
            result.append(table[rb][(cb + 1) % 8])
        elif ca == cb:
            result.append(table[(ra + 1) % 8][ca])
            result.append(table[(rb + 1) % 8][cb])
        else:
            result.append(table[ra][cb])
            result.append(table[rb][ca])
    return "".join(result)


def encrypt(plaintext: str, key: str) -> str:
    table, lookup = generate_key_data(key)
    digrams = prepare_text(plaintext)
    encrypted_body = encrypt_digrams(table, lookup, digrams)
    return encrypted_body


file_path = input("Enter path of plain text file: ").strip()
key = input("Enter key: ").strip()

if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found.")
else:
    with open(file_path, "r", encoding="utf-8") as f:
        plain_text = f.read()

    encrypted_text = encrypt(plain_text, key)

    file_name = os.path.splitext(os.path.basename(file_path))[0]
    directory = os.path.dirname(file_path)
    output_file = os.path.join(directory, f"{file_name}_Encrypted_PF.txt")

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(encrypted_text)

    print(f"Encryption successful!")
    print(f"Key used: {key}")
    print(f"Original file: {file_path}")
    print(f"Encrypted file saved: {output_file}")

### `Decryption`

In [ ]:
import os
import string

CHARS = string.ascii_uppercase + string.digits + "@#%&*()_+-={}[]:;\"'<>,.?! \n\t"


def generate_key_data(key: str) -> tuple[list[list[str]], dict[str, tuple[int, int]]]:
    key = key.upper()
    seen_chars = []
    seen_set = set()
    for ch in key + CHARS:
        if ch in CHARS and ch not in seen_set:
            seen_chars.append(ch)
            seen_set.add(ch)
    table = [seen_chars[i * 8 : (i + 1) * 8] for i in range(8)]
    lookup = {ch: (r, c) for r, row in enumerate(table) for c, ch in enumerate(row)}
    return table, lookup


def clean_playfair_artifacts(text: str) -> str:
    cleaned = []
    i = 0
    while i < len(text):
        if (
            text[i] == "X"
            and i > 0
            and i < len(text) - 1
            and text[i - 1] == text[i + 1]
            and text[i - 1] != "X"
        ):
            i += 1
            continue
        if (
            text[i] == "Y"
            and i > 0
            and i < len(text) - 1
            and text[i - 1] == "X"
            and text[i + 1] == "X"
        ):
            i += 1
            continue
        cleaned.append(text[i])
        i += 1
    result = "".join(cleaned)
    if len(result) >= 2:
        if result.endswith("X") and result[-2] != "X":
            result = result[:-1]
        elif result.endswith("Y") and result[-2] == "X":
            result = result[:-1]
    return result


def decrypt_digrams(
    table: list[list[str]], lookup: dict[str, tuple[int, int]], ciphertext: str
) -> str:
    chars = [ch for ch in ciphertext.upper() if ch in CHARS]
    if len(chars) % 2 != 0:
        chars.append("X")
    result = []
    for i in range(0, len(chars), 2):
        a, b = chars[i], chars[i + 1]
        ra, ca = lookup[a]
        rb, cb = lookup[b]
        if ra == rb:
            result.append(table[ra][(ca - 1) % 8])
            result.append(table[rb][(cb - 1) % 8])
        elif ca == cb:
            result.append(table[(ra - 1) % 8][ca])
            result.append(table[(rb - 1) % 8][cb])
        else:
            result.append(table[ra][cb])
            result.append(table[rb][ca])
    return "".join(result)


def decrypt(ciphertext: str, key: str, auto_clean: bool = True) -> str:
    table, lookup = generate_key_data(key)
    raw_plaintext = decrypt_digrams(table, lookup, ciphertext)
    final_plaintext = (
        clean_playfair_artifacts(raw_plaintext) if auto_clean else raw_plaintext
    )
    return final_plaintext


file_path = input("Enter path of encrypted text file: ").strip()
key = input("Enter key: ").strip()

if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found.")
else:
    with open(file_path, "r", encoding="utf-8") as f:
        encrypted_text = f.read()

    decrypted_text = decrypt(encrypted_text, key)

    file_name = os.path.splitext(os.path.basename(file_path))[0]
    directory = os.path.dirname(file_path)
    output_file = os.path.join(directory, f"{file_name}_Decrypted_PF.txt")

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(decrypted_text)

    print(f"Decryption successful!")
    print(f"Key used: {key}")
    print(f"Encrypted file: {file_path}")
    print(f"Decrypted file saved: {output_file}")